# Part 1: Predicting resources and latency

This notebook predicts the FPGA resources and latency of a neural network before synthesis, with the predictors included in `rule4ml`.

## The models to predict

`rule4ml` reads PyTorch and Keras models. Here's a small PyTorch classifier.

In [ ]:
import torch


class JetTagger(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.dense1 = torch.nn.Linear(16, 64)
        self.dense2 = torch.nn.Linear(64, 32)
        self.dense3 = torch.nn.Linear(32, 32)
        self.output = torch.nn.Linear(32, 5)
        self.relu = torch.nn.ReLU()
        self.softmax = torch.nn.Softmax(dim=-1)

    def forward(self, inputs):
        x = self.relu(self.dense1(inputs))
        x = self.relu(self.dense2(x))
        x = self.relu(self.dense3(x))

        return self.softmax(self.output(x))


model = JetTagger()

## Loading the default predictors

`MultiSurrogate` combines one or more predictors and merges predictions into one table. Each predictor can cover one or several targets.

`load_default_surrogates` adds the predictors included in the package, one per target.

In [ ]:
from rule4ml.surrogates import MultiSurrogate

estimator = MultiSurrogate()
estimator.load_default_surrogates()

## Predicting for chosen settings

Each HLS config sets the board and the hls4ml model settings. Predictions cover every combination of model, config and tool versions.

Here we compare reuse factors and strategies on one board, with one hls4ml and Vivado version.

In [ ]:
hls_configs = [
    {
        "board": "zcu102",
        "model": {
            "precision": "ap_fixed<8,3>",
            "reuse_factor": reuse_factor,
            "strategy": strategy,
        },
    }
    for strategy in ("Latency", "Resource")
    for reuse_factor in (1, 4, 16)
]

predictions = estimator.predict(
    model,
    hls_configs=hls_configs,
    hls4ml_versions="1.1.0",
    vivado_versions="2024.2",
)
predictions

## Reading the predictions

- **Resources:** one column per resource with its count, and a `(%)` column with its share of the board. BRAM is counted in the units of the report a predictor was trained on (BRAM18K for HLS, BRAM36K for Vivado).
- **Latency:** clock cycles and initiation interval, minimum/maximum or averages.

`predict` warns when a setting is absent from the predictors' training data. Those predictions can be less reliable.

In [ ]:
columns = ["Strategy", "Reuse Factor", "Clock Period", "LUT", "LUT (%)", "CYCLES_MAX"]
summary = predictions[columns].copy()
summary

Without `hls_configs` or versions, `predict` sweeps the boards, strategies, clock periods and tool versions of the training data, with default precisions and reuse factors. This gives a full picture at the cost of a much larger table.

In [ ]:
overview = estimator.predict(model)
print(f"{len(overview)} rows total")
overview

## Saving the predictions

Predictions of `MultiSurrogate` are a pandas `DataFrame`, so they export to CSV, HTML, LaTeX and other formats.

In [ ]:
predictions.to_csv("predictions.csv", index=False)

## Keras models

Keras models need `pip install "rule4ml[keras]"` and are passed to `predict` the same way. The [README](https://github.com/IMPETUS-UdeS/rule4ml#usage) has a Keras example.

## Using a single predictor

A predictor can also be loaded on its own, here the default LUT predictor. A single predictor needs the tool versions and returns a NumPy array with one row per combination of model, config and versions.

In [ ]:
import os

import rule4ml
from rule4ml.surrogates import TorchSurrogate

weights_dir = os.path.join(
    os.path.dirname(rule4ml.__file__), "models", "weights", "default"
)
lut_predictor = TorchSurrogate()
lut_predictor.load(
    os.path.join(weights_dir, "LUT.config.json"),
    os.path.join(weights_dir, "LUT.weights.pt"),
)

lut_predictor.predict(
    [model], hls_configs, hls4ml_versions=["1.1.0"], vivado_versions=["2024.2"]
)